In [ ]:
import jax
import jax.numpy as jnp
from exciting_environments import EnvironmentRegistry
from exciting_environments.utils import MinMaxNormalization
from exciting_environments.pmsm.motor_parameters import MotorVariant

In [ ]:
pend_env=EnvironmentRegistry.PENDULUM.make()

## Step and Simulate ahead

In [ ]:
key=jax.random.PRNGKey(1234)
obs, state = pend_env.reset(key)

In [ ]:
obs,states,last_state=pend_env.sim_ahead(state,jnp.ones((4,1)))
obs

In [ ]:
pend_env.generate_rew_trunc_term_ahead(states,jnp.ones((4,1)))

In [ ]:
key=jax.random.PRNGKey(1234)
obs, state = pend_env.reset()
generated_observations = []
generated_actions= []
generated_observations.append(obs)
for i in range(4):
    key,subkey= jax.random.split(key)
    action = jnp.ones(2)#jax.random.uniform(subkey,(2,),minval=-1,maxval=1)
    obs, state = pend_env.step(state, action)
    generated_actions.append(action)
    generated_observations.append(obs)

In [ ]:
generated_observations

### Vmapped

In [ ]:
env1=EnvironmentRegistry.PENDULUM.make(static_params={"g": jnp.array(9.81), "l": jnp.array(1.0),  "m": jnp.array(1.0)})
env2=EnvironmentRegistry.PENDULUM.make(static_params={"g": jnp.array(9.81), "l": jnp.array(2.0),  "m": jnp.array(1.0)})
envs = [env1,env2]
batched_envs = EnvironmentRegistry.batch_envs(envs)
batch_size= 2
#batched_envs = EnvironmentRegistry.PENDULUM.make(batch_size=batch_size)

In [ ]:
keys = jax.random.split(jax.random.PRNGKey(0), batch_size)
obs, states = batched_envs.vmap_reset(keys)
print(obs)

In [ ]:
state_in=batched_envs.vmap_generate_state_from_observation(obs,keys)
state_in

In [ ]:
actions = jnp.ones((batch_size,1))
obs, states = batched_envs.vmap_reset()
next_obs, next_states = batched_envs.vmap_step(states, actions)
print(next_obs)

In [ ]:
keys = jax.random.split(jax.random.PRNGKey(0), batch_size)
obs, states = batched_envs.vmap_reset()
actions = jnp.ones((batch_size,4,1))
next_obs, next_states, last_state = batched_envs.vmap_sim_ahead(states, actions)
print(next_obs)

In [ ]:
batched_envs.vmap_generate_rew_trunc_term_ahead(next_states, actions)